In [1]:
from datetime import datetime

print(datetime.now())

2026-10-08 11:16:15.479858


In [2]:
import pandas as pd

In [3]:
orders = pd.read_csv("../data/orders.csv")
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery


In [4]:
category_order_summary = (
    orders
    .groupby(['category'], as_index = False)
    .agg(
        gmv = ("gmv", "sum"),
        order_count = ("order_id", "nunique")
    )
    .sort_values(
        by = ["gmv"],
        ascending = [False]
    )
)

total_gmv = category_order_summary["gmv"].sum()

category_order_summary["pct_gmv"] = 100*category_order_summary["gmv"]/total_gmv

print(datetime.now())
category_order_summary

2026-10-08 11:16:16.742244


,category,gmv,order_count,pct_gmv
0,dairy,6892094.70,16146,16.757972
1,grocery,6876796.20,16005,16.720774
5,snacks,6862504.21,15901,16.686023
2,household,6836939.41,15979,16.623863
3,personal_care,6833477.47,16020,16.615446
4,pharmacy,6825448.00,15969,16.595922


In [5]:
orders["order_ts"] = pd.to_datetime(orders['order_ts'])
orders["order_hour"] = orders["order_ts"].dt.hour
orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_hour
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy,10
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy,7
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy,15
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy,13
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery,14


In [6]:
hour_order_summary = (
    orders
    .groupby(['order_hour'], as_index = False)
    .agg(
        order_count = ("order_id", "nunique")
    )
    .sort_values(
        by = ["order_hour"],
        ascending = [True]
    )
)

print(datetime.now())
hour_order_summary

2026-10-08 11:16:16.943342


,order_hour,order_count
0,7,5720
1,8,5737
2,9,5623
3,10,5750
4,11,5646
5,12,5504
6,13,5574
7,14,5759
8,15,5621
9,16,5495


In [7]:
city_hour_order_summary = (
    orders
    .groupby(['city', 'order_hour'], as_index = False)
    .agg(
        order_count = ("order_id", "nunique")
    )
    .sort_values(
        by = ["city", 'order_count'],
        ascending = [True, False]
    )
)

city_hour_order_summary["rank"] = (
    city_hour_order_summary
    .groupby(['city'])['order_count']
    .rank(
        method = 'first',
        ascending = False
    )
)

print(datetime.now())
city_hour_order_summary

2026-10-08 11:16:17.028920


,city,order_hour,order_count,rank
15,Bengaluru,22,1477,1.0
10,Bengaluru,17,1473,2.0
13,Bengaluru,20,1473,3.0
12,Bengaluru,19,1450,4.0
7,Bengaluru,14,1437,5.0
...,...,...,...,...
89,Pune,11,567,13.0
87,Pune,9,563,14.0
90,Pune,12,560,15.0
95,Pune,17,557,16.0


In [8]:
peak_city_hours = city_hour_order_summary[city_hour_order_summary['rank'] == 1]
print(datetime.now())
peak_city_hours

2026-10-08 11:16:17.066544


,city,order_hour,order_count,rank
15,Bengaluru,22,1477,1.0
24,Delhi,14,1407,1.0
48,Gurugram,21,608,1.0
59,Hyderabad,15,622,1.0
69,Mumbai,8,1236,1.0
85,Pune,7,614,1.0


In [9]:
orders["order_year"] = orders["order_ts"].dt.year
orders["order_month"] = orders["order_ts"].dt.month

orders_2026_monthly_summary = (
    orders[orders['order_year'] == 2026]
    .groupby(['order_month'], as_index = False)
    .agg(
        gmv = ("gmv", "sum")
    )
    .sort_values(by = ['order_month'], ascending = [True])
)

print(datetime.now())
orders_2026_monthly_summary

2026-10-08 11:16:17.174267


,order_month,gmv
0,1,2110794.11
1,2,2176684.44
2,3,2775810.15
3,4,3065139.39
4,5,3645427.05
5,6,4086301.97
6,7,4518165.14
7,8,4449698.57
8,9,4129042.64


In [10]:
users = pd.read_csv("../data/users.csv")
users.head()

,user_id,signup_date,city,acquisition_channel
0,1,2026-05-26,Bengaluru,referral
1,2,2025-12-04,Pune,search
2,3,2026-01-05,Hyderabad,organic
3,4,2026-04-30,Bengaluru,search
4,5,2025-11-09,Delhi,organic


In [11]:
orders = orders.merge(users[['user_id', 'acquisition_channel']], on = ['user_id'], how = 'left')

city_acquisition_channel_order_summary = (
    orders
    .groupby(['city', 'acquisition_channel'], as_index = False)
    .agg(
        gmv = ("gmv", "sum"),
        order_count = ("order_id", "nunique")
    )
)

city_acquisition_channel_order_summary["aov"] = city_acquisition_channel_order_summary["gmv"]/city_acquisition_channel_order_summary["order_count"]

print(datetime.now())
city_acquisition_channel_order_summary



2026-10-08 11:16:17.588117


,city,acquisition_channel,gmv,order_count,aov
0,Bengaluru,organic,4687703.94,10870,431.251512
1,Bengaluru,paid_social,2561110.91,5899,434.160181
2,Bengaluru,referral,1537994.16,3569,430.931398
3,Bengaluru,search,1605913.48,3835,418.751885
4,Delhi,organic,4329575.32,10168,425.804024
5,Delhi,paid_social,2574558.45,5999,429.164602
6,Delhi,referral,1469985.41,3450,426.082728
7,Delhi,search,1406954.59,3282,428.688175
8,Gurugram,organic,1746985.58,4116,424.437702
9,Gurugram,paid_social,1093328.32,2576,424.428696


In [12]:
pivot_df = city_acquisition_channel_order_summary.pivot(
    index="acquisition_channel",
    columns="city",
    values="aov"
)

print(datetime.now())
pivot_df

2026-10-08 11:16:17.697660


city,Bengaluru,Delhi,Gurugram,Hyderabad,Mumbai,Pune
acquisition_channel,,,,,,
organic,431.251512,425.804024,424.437702,432.105537,428.830007,427.164133
paid_social,434.160181,429.164602,424.428696,431.516378,429.483679,424.964436
referral,430.931398,426.082728,437.701241,421.023618,423.006915,420.473362
search,418.751885,428.688175,432.433505,439.003006,425.367327,429.328295


In [13]:
city_store_order_summary = (
    orders
    .groupby(['city', 'store_id'], as_index = False)
    .agg(
        gmv = ("gmv", "sum")
    )
    .sort_values(
        by = ["city", 'gmv'],
        ascending = [True, False]
    )
)

city_store_order_summary["rank"] = (
    city_store_order_summary
    .groupby(['city'])['gmv']
    .rank(
        method = 'first',
        ascending = False
    )
)

print(datetime.now())
city_store_order_summary

2026-10-08 11:16:17.895256


,city,store_id,gmv,rank
5,Bengaluru,BEN-06,1369130.47,1.0
1,Bengaluru,BEN-02,1322807.08,2.0
2,Bengaluru,BEN-03,1310569.50,3.0
3,Bengaluru,BEN-04,1310135.24,4.0
4,Bengaluru,BEN-05,1301647.30,5.0
6,Bengaluru,BEN-07,1261571.60,6.0
7,Bengaluru,BEN-08,1259377.36,7.0
0,Bengaluru,BEN-01,1257483.94,8.0
14,Delhi,DEL-07,1251133.66,1.0
9,Delhi,DEL-02,1240592.45,2.0


In [14]:
top_three_city_stores = city_store_order_summary[city_store_order_summary["rank"] <= 3]
print(datetime.now())
top_three_city_stores

2026-10-08 11:16:18.002608


,city,store_id,gmv,rank
5,Bengaluru,BEN-06,1369130.47,1.0
1,Bengaluru,BEN-02,1322807.08,2.0
2,Bengaluru,BEN-03,1310569.50,3.0
14,Delhi,DEL-07,1251133.66,1.0
9,Delhi,DEL-02,1240592.45,2.0
8,Delhi,DEL-01,1229749.16,3.0
18,Gurugram,GUR-03,537811.30,1.0
16,Gurugram,GUR-01,530179.68,2.0
21,Gurugram,GUR-06,529912.98,3.0
28,Hyderabad,HYD-05,556345.80,1.0


In [15]:
users['signup_date'] = pd.to_datetime(users['signup_date'])
orders['order_date'] = orders['order_ts'].dt.date

user_first_order_date = (
    orders
    .groupby(['user_id'], as_index = False)
    .agg(
        first_order_date = ("order_date", "min")
    )
)

users = users.merge(user_first_order_date, on = 'user_id', how = 'left')

In [16]:
users["signup_date"] = pd.to_datetime(users["signup_date"])
users["first_order_date"] = pd.to_datetime(users["first_order_date"])

users["days_to_first_order"] = (
    users["first_order_date"] - users["signup_date"]
).dt.days

users.head()

,user_id,signup_date,city,acquisition_channel,first_order_date,days_to_first_order
0,1,2026-05-26,Bengaluru,referral,2026-06-04,9.0
1,2,2025-12-04,Pune,search,2026-02-18,76.0
2,3,2026-01-05,Hyderabad,organic,2026-03-26,80.0
3,4,2026-04-30,Bengaluru,search,2026-06-17,48.0
4,5,2025-11-09,Delhi,organic,2025-12-05,26.0


In [17]:
median_by_channel = (
    users
    .groupby(["acquisition_channel"], as_index = False)
    .agg(
        converted_users = ("user_id", "count"),
        not_converted_users = ("days_to_first_order", "count"),
        median_days_to_first_order = ("days_to_first_order", "median")
    )
)

median_by_channel["pct_not_converted_users"] = 100*median_by_channel['not_converted_users']/(median_by_channel['converted_users'] + median_by_channel['not_converted_users'])

print(datetime.now())
median_by_channel

2026-10-08 11:16:21.565006


,acquisition_channel,converted_users,not_converted_users,median_days_to_first_order,pct_not_converted_users
0,organic,9052,7751,47.0,46.128667
1,paid_social,4975,4282,49.0,46.256887
2,referral,2962,2543,50.0,46.194369
3,search,3011,2585,46.0,46.193710


In [18]:
orders['delivery_min_more_than_25'] = orders["delivery_min"] > 25

city_store_order_summary = (
    orders
    .groupby(['city', 'store_id'], as_index = False)
    .agg(
        total_orders = ("order_id", "count"),
        flagged_orders = ("delivery_min_more_than_25", "sum")
    )
    .sort_values(
        by = ["city", 'store_id'],
        ascending = [True, True]
    )
)

city_store_order_summary["total_city_orders"] = (
    city_store_order_summary
    .groupby("city")["total_orders"]
    .transform("sum")
)

city_store_order_summary["flagged_city_orders"] = (
    city_store_order_summary
    .groupby("city")["flagged_orders"]
    .transform("sum")
)

city_store_order_summary["pct_flagged_orders"] = 100*city_store_order_summary['flagged_orders']/city_store_order_summary['total_orders']

city_store_order_summary["pct_flagged_city_orders"] = 100*city_store_order_summary['flagged_city_orders']/city_store_order_summary['total_city_orders']

city_store_order_summary["is_store_performing_worse_than_city_avg"] = city_store_order_summary["pct_flagged_orders"] > city_store_order_summary["pct_flagged_city_orders"]

print(datetime.now())
city_store_order_summary

2026-10-08 11:16:21.664927


,city,store_id,total_orders,flagged_orders,total_city_orders,flagged_city_orders,pct_flagged_orders,pct_flagged_city_orders,is_store_performing_worse_than_city_avg
0,Bengaluru,BEN-01,3017,146,24173,1246,4.839244,5.154511,False
1,Bengaluru,BEN-02,3114,163,24173,1246,5.234425,5.154511,True
2,Bengaluru,BEN-03,3033,181,24173,1246,5.967689,5.154511,True
3,Bengaluru,BEN-04,3013,148,24173,1246,4.912048,5.154511,False
4,Bengaluru,BEN-05,2981,143,24173,1246,4.797048,5.154511,False
5,Bengaluru,BEN-06,3121,153,24173,1246,4.902275,5.154511,False
6,Bengaluru,BEN-07,2939,154,24173,1246,5.239878,5.154511,True
7,Bengaluru,BEN-08,2955,158,24173,1246,5.346870,5.154511,True
8,Delhi,DEL-01,2870,144,22899,1121,5.017422,4.895410,True
9,Delhi,DEL-02,2919,134,22899,1121,4.590613,4.895410,False


In [19]:
orders["order_date"] = pd.to_datetime(orders["order_date"])

orders["day_of_week"] = orders["order_date"].dt.day_name()

orders["is_weekend"] = orders['day_of_week'].isin(["Saturday", "Sunday"])

orders.head()

,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_hour,order_year,order_month,acquisition_channel,order_date,delivery_min_more_than_25,day_of_week,is_weekend
0,1,10700,MUM-07,Mumbai,2025-01-02 10:44:00,5,155.93,20.9,dairy,10,2025,1,search,2025-01-02,False,Thursday,False
1,2,8110,DEL-03,Delhi,2025-01-06 07:09:00,2,123.86,11.9,pharmacy,7,2025,1,organic,2025-01-06,False,Monday,False
2,3,15650,PUN-01,Pune,2025-01-06 15:27:00,2,184.46,16.8,pharmacy,15,2025,1,paid_social,2025-01-06,False,Monday,False
3,4,1908,DEL-07,Delhi,2025-01-08 13:06:00,8,395.88,24.6,pharmacy,13,2025,1,paid_social,2025-01-08,False,Wednesday,False
4,5,2751,BEN-07,Bengaluru,2025-01-08 14:57:00,5,199.17,21.8,grocery,14,2025,1,referral,2025-01-08,False,Wednesday,False


In [20]:
category_is_weekend_order_summary = (
    orders
    .groupby(['category', 'is_weekend'], as_index = False)
    .agg(
        gmv = ("gmv", "sum"),
        order_count = ("order_id", "nunique")
    )
)

category_is_weekend_order_summary["aov"] = category_is_weekend_order_summary["gmv"]/category_is_weekend_order_summary["order_count"]

print(datetime.now())
category_is_weekend_order_summary



2026-10-08 11:16:21.896971


,category,is_weekend,gmv,order_count,aov
0,dairy,False,4909361.61,11437,429.252567
1,dairy,True,1982733.09,4709,421.051835
2,grocery,False,4953432.53,11534,429.463545
3,grocery,True,1923363.67,4471,430.186462
4,household,False,4929634.64,11494,428.887649
5,household,True,1907304.77,4485,425.263048
6,personal_care,False,4867487.24,11450,425.108056
7,personal_care,True,1965990.23,4570,430.194799
8,pharmacy,False,4882425.21,11408,427.982575
9,pharmacy,True,1943022.79,4561,426.008066


In [21]:
pivot_df = category_is_weekend_order_summary.pivot(
    index="category",
    columns="is_weekend",
    values="aov"
)

print(datetime.now())
pivot_df

2026-10-08 11:16:21.932717


is_weekend,False,True
category,,
dairy,429.252567,421.051835
grocery,429.463545,430.186462
household,428.887649,425.263048
personal_care,425.108056,430.194799
pharmacy,427.982575,426.008066
snacks,432.214121,430.011398


In [22]:
# orders["user_order_rank"] = (
#     orders
#     .groupby("user_id")["gmv"]
#     .rank(
#         method="first",
#         ascending=False
#     )
# )

largest_orders = orders.loc[orders.groupby("user_id")["gmv"].idxmax()]
print(datetime.now())
largest_orders

2026-10-08 11:16:21.990390


,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_hour,order_year,order_month,acquisition_channel,order_date,delivery_min_more_than_25,day_of_week,is_weekend
70176,70177,1,BEN-04,Bengaluru,2026-07-14 18:08:00,5,539.01,14.8,pharmacy,18,2026,7,referral,2026-07-14,False,Tuesday,False
51101,51102,2,PUN-01,Pune,2026-05-15 08:26:00,5,474.62,12.2,personal_care,8,2026,5,search,2026-05-15,False,Friday,False
83152,83153,3,HYD-05,Hyderabad,2026-08-22 13:42:00,8,1147.82,10.1,household,13,2026,8,organic,2026-08-22,False,Saturday,True
61093,61094,4,BEN-02,Bengaluru,2026-06-17 20:30:00,6,464.25,11.4,personal_care,20,2026,6,search,2026-06-17,False,Wednesday,False
21893,21894,5,DEL-08,Delhi,2025-12-20 10:34:00,5,704.40,9.5,household,10,2025,12,organic,2025-12-20,False,Saturday,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58109,58110,19996,MUM-03,Mumbai,2026-06-08 12:39:00,10,394.66,9.5,household,12,2026,6,organic,2026-06-08,False,Monday,False
63187,63188,19997,DEL-08,Delhi,2026-06-23 21:59:00,7,716.82,13.0,household,21,2026,6,search,2026-06-23,False,Tuesday,False
27987,27988,19998,BEN-02,Bengaluru,2026-01-28 10:59:00,7,656.89,18.7,personal_care,10,2026,1,referral,2026-01-28,False,Wednesday,False
37720,37721,19999,MUM-05,Mumbai,2026-03-20 12:30:00,7,1367.09,15.7,pharmacy,12,2026,3,paid_social,2026-03-20,False,Friday,False


In [23]:
largest_orders = largest_orders.sort_values(by = ["gmv"], ascending = [False])
print(datetime.now())
largest_orders.head()

2026-10-08 11:16:22.061777


,order_id,user_id,store_id,city,order_ts,items,gmv,delivery_min,category,order_hour,order_year,order_month,acquisition_channel,order_date,delivery_min_more_than_25,day_of_week,is_weekend
44640,44641,8766,DEL-03,Delhi,2026-04-20 11:13:00,12,5032.83,12.6,grocery,11,2026,4,paid_social,2026-04-20,False,Monday,False
17810,17811,14033,GUR-08,Gurugram,2025-11-19 23:40:00,9,4702.54,34.3,snacks,23,2025,11,organic,2025-11-19,True,Wednesday,False
56585,56586,1455,DEL-05,Delhi,2026-06-03 10:51:00,9,4684.62,16.5,snacks,10,2026,6,paid_social,2026-06-03,False,Wednesday,False
90188,90189,13349,BEN-03,Bengaluru,2026-09-12 10:20:00,10,4654.39,10.0,grocery,10,2026,9,organic,2026-09-12,False,Saturday,True
54852,54853,8041,MUM-02,Mumbai,2026-05-28 12:29:00,11,4558.78,21.5,pharmacy,12,2026,5,organic,2026-05-28,False,Thursday,False


In [24]:
user_category_order_summary = (
    orders
    .groupby(["user_id", 'city'], as_index = False)
    .agg(
        order_categories = ('category', 'nunique')
    )
)

user_category_order_summary['flag'] = user_category_order_summary['order_categories'] >= 4

city_flagged_user_summary = (
    user_category_order_summary
    .groupby(['city'], as_index = False)
    .agg(
        user_count = ("flag", "sum")
    )
    .sort_values(by = ['city'], ascending = [True])
)

print(datetime.now())
city_flagged_user_summary

2026-10-08 11:16:22.246385


,city,user_count
0,Bengaluru,1819
1,Delhi,1736
2,Gurugram,717
3,Hyderabad,718
4,Mumbai,1489
5,Pune,764
